# 1. Introduction

CNN based models for galaxy classification.

# 2. Dataset 

The objective is to classify the given galaxy images.


The dataset consists of 17,736 images, each being a 256x256 RGB matrix, categorized into 10 classes.

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
%pip install e2cnn

In [ ]:
import os
import copy
import math
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

- **`Galaxy10_DECals.h5`** — your training data  

In [ ]:
# ── Load the training data ───────────────────────────────────
import h5py

train_h5_candidates = [
    './Galaxy10_DECals.h5',
    'Galaxy10_DECals.h5',
    '/home/yangz2/code/space_universe/data/A5/Galaxy10_DECals.h5',
]

train_h5 = None
for cand in train_h5_candidates:
    if os.path.exists(cand):
        train_h5 = cand
        break

if train_h5 is None:
    raise FileNotFoundError('Galaxy10_DECals.h5 not found. Please place it in the current working directory.')

with h5py.File(train_h5, 'r') as f:
    images_all = f['images'][:]   # (N, 256, 256, 3)
    labels_all = f['ans'][:]      # (N,)

print(f'Loaded training pool from: {train_h5}')
print(f'Images shape: {images_all.shape}')
print(f'Labels shape: {labels_all.shape}')

In [ ]:
# ── Stratified split: local train / local validation ─────────
# The official held-out test set is Galaxy10_DECals_20pct.h5 and is NOT used here.
X_train, X_val, y_train, y_val = train_test_split(
    images_all, labels_all,
    test_size=0.20,
    random_state=42,
    stratify=labels_all
)

print(f'Train set : {X_train.shape}')
print(f'Val set   : {X_val.shape}')

train_counts = np.bincount(y_train, minlength=10)
val_counts   = np.bincount(y_val, minlength=10)

print('\nPer-class counts (train):', train_counts)
print('Per-class counts (val)  :', val_counts)

In [ ]:
galaxy_classes = [
    'Disturbed Galaxies',
    'Merging Galaxies',
    'Round Smooth Galaxies',
    'In-between Round Smooth Galaxies',
    'Cigar Shaped Smooth Galaxies',
    'Barred Spiral Galaxies',
    'Unbarred Tight Spiral Galaxies',
    'Unbarred Loose Spiral Galaxies',
    'Edge-on Galaxies without Bulge',
    'Edge-on Galaxies with Bulge',
]

# Galaxy10 dataset (17736 images)
# ├── Class 0 (1081 images): Disturbed Galaxies
# ├── Class 1 (1853 images): Merging Galaxies
# ├── Class 2 (2645 images): Round Smooth Galaxies
# ├── Class 3 (2027 images): In-between Round Smooth Galaxies
# ├── Class 4 ( 334 images): Cigar Shaped Smooth Galaxies
# ├── Class 5 (2043 images): Barred Spiral Galaxies
# ├── Class 6 (1829 images): Unbarred Tight Spiral Galaxies
# ├── Class 7 (2628 images): Unbarred Loose Spiral Galaxies
# ├── Class 8 (1423 images): Edge-on Galaxies without Bulge
# └── Class 9 (1873 images): Edge-on Galaxies with Bulge

print('Unique classes in training set:', np.unique(y_train))

In [ ]:
# Visualise sample training images

import random

fig = plt.figure(figsize=(20, 20))
for i in range(25):
    idx = i + random.randint(0, 100)
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_train[idx])
    plt.title(galaxy_classes[y_train[idx]], fontsize=8)
    fig.tight_layout(pad=3.0)
plt.show()

In [ ]:
# ── PyTorch Dataset wrapper ──────────────────────────────────

class GalaxyDataset(Dataset):
    def __init__(self, images, labels=None, transform=None):
        """
        images: NumPy array of shape (N, 256, 256, 3), dtype uint8
        labels: NumPy array of shape (N,), or None for inference
        transform: torchvision transforms pipeline
        """
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]  # HWC uint8
        if self.transform is not None:
            image = self.transform(image)
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        if self.labels is None:
            return image
        return image, int(self.labels[idx])

# 3. Methodology

## 3.1 Appetizer

### 3.1.1 Simple CNN

We use a 3-layer CNN network, with batch normalisation layers.

In [ ]:
# ── Define the improved model source as a string ─────────────
model_source = """
import numpy as np
import torch
import torch.nn as nn

from e2cnn import gspaces
from e2cnn import nn as enn
import e2cnn.kernels.irreps_basis as _ib
import e2cnn.kernels.utils as _ku


def _patch_e2cnn_numpy_compat():
    # e2cnn 0.2.x uses a few NumPy 1.x-only calls; patch them so the notebook
    # remains runnable in newer environments too.
    patch_targets = [
        'R2FlipsSolution',
        'R2DiscreteRotationsSolution',
        'R2FlipsDiscreteRotationsSolution',
        'R2ContinuousRotationsSolution',
    ]

    def _patched_hash(self):
        mu = self.mu.tobytes() if hasattr(self.mu, 'tobytes') else self.mu
        gamma = self.gamma.tobytes() if hasattr(self.gamma, 'tobytes') else self.gamma
        return hash(self.in_irrep) + hash(self.out_irrep) + hash(mu) + hash(gamma)

    for _name in patch_targets:
        if hasattr(_ib, _name):
            getattr(_ib, _name).__hash__ = _patched_hash

    def _psi(theta, k=1, gamma=0.0, out=None):
        if isinstance(theta, float):
            theta = np.array(theta)
        k = np.asarray(k).reshape(-1, 1)
        gamma = np.asarray(gamma).reshape(-1, 1)
        theta = np.asarray(theta).reshape(1, -1)

        x = k * theta + gamma
        cos, sin = np.cos(x), np.sin(x)

        if out is None:
            out = np.empty((2, 2, x.shape[0], x.shape[-1]))

        out[0, 0, ...] = cos
        out[0, 1, ...] = -sin
        out[1, 0, ...] = sin
        out[1, 1, ...] = cos
        return out

    def _chi(s, out=None):
        s = -1 * (s > 0) + (s <= 0)
        s = np.asarray(s).reshape(-1)

        if out is None:
            out = np.empty((2, 2, s.shape[0]))

        out[0, 0, ...] = 1
        out[0, 1, ...] = 0
        out[1, 0, ...] = 0
        out[1, 1, ...] = s
        return out

    def _psichi(theta, s, k=1, gamma=0.0, out=None):
        if isinstance(theta, float):
            theta = np.array(theta)

        s = -1 * (s > 0) + (s <= 0)
        s = np.asarray(s).reshape(-1, 1)
        k = np.asarray(k).reshape(-1, 1)
        gamma = np.asarray(gamma).reshape(-1, 1)
        theta = np.asarray(theta).reshape(1, -1)

        x = k * theta + gamma
        cos, sin = np.cos(x), np.sin(x)

        if out is None:
            out = np.empty((2, 2, x.shape[0], x.shape[-1]))

        out[0, 0, ...] = cos
        out[0, 1, ...] = -s * sin
        out[1, 0, ...] = sin
        out[1, 1, ...] = s * cos
        return out

    _ku.psi = _psi
    _ku.chi = _chi
    _ku.psichi = _psichi
    _ib.psi = _psi
    _ib.chi = _chi
    _ib.psichi = _psichi


_patch_e2cnn_numpy_compat()


class GalaxyModel(nn.Module):
    def __init__(self, num_classes=10, base_width=16, dropout=0.15):
        super().__init__()

        # Lightweight D4-equivariant E(2)-GCNN:
        # - rotations by multiples of 90°
        # - flips / mirror symmetry
        self.r2_act = gspaces.FlipRot2dOnR2(4)
        self.input_type = enn.FieldType(self.r2_act, 3 * [self.r2_act.trivial_repr])

        c1 = enn.FieldType(self.r2_act, base_width * [self.r2_act.regular_repr])
        c2 = enn.FieldType(self.r2_act, (base_width * 2) * [self.r2_act.regular_repr])
        c3 = enn.FieldType(self.r2_act, (base_width * 3) * [self.r2_act.regular_repr])
        c4 = enn.FieldType(self.r2_act, (base_width * 4) * [self.r2_act.regular_repr])

        self.block1 = enn.SequentialModule(
            enn.R2Conv(self.input_type, c1, kernel_size=7, padding=3, bias=False),
            enn.InnerBatchNorm(c1),
            enn.ReLU(c1, inplace=True),
            enn.PointwiseMaxPool(c1, 2),
        )
        self.block2 = enn.SequentialModule(
            enn.R2Conv(c1, c2, kernel_size=5, padding=2, bias=False),
            enn.InnerBatchNorm(c2),
            enn.ReLU(c2, inplace=True),
            enn.PointwiseMaxPool(c2, 2),
        )
        self.block3 = enn.SequentialModule(
            enn.R2Conv(c2, c3, kernel_size=5, padding=2, bias=False),
            enn.InnerBatchNorm(c3),
            enn.ReLU(c3, inplace=True),
            enn.PointwiseMaxPool(c3, 2),
        )
        self.block4 = enn.SequentialModule(
            enn.R2Conv(c3, c4, kernel_size=3, padding=1, bias=False),
            enn.InnerBatchNorm(c4),
            enn.ReLU(c4, inplace=True),
        )

        self.group_pool = enn.GroupPooling(c4)
        pooled_channels = self.group_pool.out_type.size

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(pooled_channels, 96),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(96, num_classes),
        )

        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = x.float()
        if x.max() > 1.5:
            x = x / 255.0
        x = (x - self.mean) / self.std

        x = enn.GeometricTensor(x, self.input_type)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.group_pool(x)
        x = self.avgpool(x.tensor)
        return self.head(x)
"""

In [ ]:
# ── Build the improved model class from model_source ─────────
exec(model_source, globals())
print('GalaxyModel is ready.')

In [ ]:

# ── Transforms, datasets, sampler, and dataloaders ──────────
IMG_SIZE = 128
BATCH_SIZE = 32

class RandomRotate90:
    def __call__(self, img):
        k = torch.randint(0, 4, (1,)).item()
        return transforms.functional.rotate(img, angle=90 * k)

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    RandomRotate90(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = GalaxyDataset(X_train, y_train, transform=train_transform)
val_dataset   = GalaxyDataset(X_val, y_val, transform=val_transform)

class_counts = np.bincount(y_train, minlength=10)
sample_weights = 1.0 / np.sqrt(class_counts[y_train])
sample_weights = torch.as_tensor(sample_weights, dtype=torch.double)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f'Train samples: {len(train_dataset)}')
print(f'Val samples  : {len(val_dataset)}')
print(f'Class counts : {class_counts}')


In [ ]:

# ── Initialise model, loss, optimiser ───────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_kwargs = dict(num_classes=10, base_width=16, dropout=0.15)
model = GalaxyModel(**model_kwargs).to(device)

class_counts = np.bincount(y_train, minlength=10)
class_weights = 1.0 / np.sqrt(class_counts.astype(np.float32))
class_weights = class_weights / class_weights.mean()
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.02)
optimizer = optim.AdamW(model.parameters(), lr=2.0e-3, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=28, eta_min=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print(f'Device       : {device}')
print(f'Class weights: {class_weights.detach().cpu().numpy()}')
print(f'Parameters   : {sum(p.numel() for p in model.parameters()):,}')


In [ ]:

# ── Training loop (select best model by Macro-F1) ───────────
def predict_with_tta(model, images):
    # For this lightweight D4-equivariant model, keep evaluation simple.
    return model(images)

num_epochs = 28
best_f1 = -1.0
best_acc = -1.0
best_state = None
patience = 8
epochs_no_improve = 0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_acc': [],
    'val_macro_f1': []
}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for imgs, lbls in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(imgs)
            loss = criterion(outputs, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * lbls.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == lbls).sum().item()
        train_total += lbls.size(0)

    train_loss = running_loss / train_total
    train_acc = train_correct / train_total

    model.eval()
    y_true_val, y_pred_val = [], []

    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            lbls = lbls.to(device, non_blocking=True)

            outputs = predict_with_tta(model, imgs)
            preds = outputs.argmax(dim=1)

            y_true_val.extend(lbls.cpu().numpy())
            y_pred_val.extend(preds.cpu().numpy())

    val_acc = accuracy_score(y_true_val, y_pred_val)
    val_macro_f1 = f1_score(y_true_val, y_pred_val, average='macro')

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_macro_f1'].append(val_macro_f1)

    print(
        f"Epoch [{epoch+1:02d}/{num_epochs}]  "
        f"Train Loss: {train_loss:.4f}  "
        f"Train Acc: {train_acc:.4f}  "
        f"Val Acc: {val_acc:.4f}  "
        f"Val Macro-F1: {val_macro_f1:.4f}"
    )

    scheduler.step()

    if val_macro_f1 > best_f1:
        best_f1 = val_macro_f1
        best_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
        print(f"  -> Best model updated (Val Macro-F1 = {best_f1:.4f}, Val Acc = {best_acc:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print('Early stopping triggered.')
            break

if best_state is None:
    raise RuntimeError('Training did not produce a valid best_state.')

model.load_state_dict(best_state)
print(f'Best validation accuracy : {best_acc*100:.2f}%')
print(f'Best validation macro-F1 : {best_f1:.4f}')

checkpoint = {
    'state_dict': model.state_dict(),
    'model_source': model_source,
    'model_class': 'GalaxyModel',
    'num_classes': 10,
    'model_kwargs': model_kwargs,
}
torch.save(checkpoint, './MyImprovedModel.pth')
print('Saved checkpoint: ./MyImprovedModel.pth')


In [ ]:

# ── Save the inference preprocessing file used by run_inference.py ───────────
preprocessing_py = """
from torchvision import transforms

inference_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])
"""

with open('my_preprocessing.py', 'w', encoding='utf-8') as f:
    f.write(preprocessing_py)

print('Saved preprocessing file: ./my_preprocessing.py')


In [ ]:
# ── Rebuild the class once from the saved source (sanity check) ──────────────
ns = {}
exec(model_source, ns)
ModelClass = ns['GalaxyModel']
print('Checkpoint rebuild class:', ModelClass.__name__)

### 3.1.2 Evaluation on Local Validation Set

We use precision, recall, f1-score, and support to evaluate the model on our **local validation split**.

In [ ]:

import torch
import torch.nn.functional as F

# ── Evaluate the saved model on the local validation set ─────────────────────
checkpoint = torch.load('./MyImprovedModel.pth', map_location=device)

ns = {}
exec(checkpoint['model_source'], ns)
ModelClass = ns[checkpoint['model_class']]
model_eval = ModelClass(**checkpoint.get('model_kwargs', {'num_classes': checkpoint['num_classes']}))
model_eval.load_state_dict(checkpoint['state_dict'])
model_eval = model_eval.to(device)
model_eval.eval()

y_true = []
y_pred = []
y_scores = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = predict_with_tta(model_eval, images)
        probs = F.softmax(outputs, dim=1)
        predicted = probs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_scores.extend(probs.cpu().numpy())

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')

print(f'Validation Accuracy : {acc*100:.2f}%')
print(f'Validation Macro-F1 : {macro_f1:.4f}')

print('
Classification Report:')
print(classification_report(y_true, y_pred, target_names=galaxy_classes, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.title('Local Validation Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.plot(history['val_macro_f1'], label='Val Macro-F1')
plt.legend()
plt.title('Training Curves')
plt.xlabel('Epoch')
plt.show()



This final version deliberately simplifies the recipe:

- **Traditional augmentation only**: random 90° rotations + horizontal / vertical flips
- **Lightweight D4-equivariant E(2)-GCNN** instead of a heavier CNN/Transformer backbone
- **Moderate parameter count** (well below the earlier ConvNeXt-based versions)
- **Class-aware sampling** and **weighted cross-entropy** for long-tail classes
- **No MixUp, no handcrafted feature branch, no heavy test-time augmentation**

The key idea is to make the model itself respect the most relevant symmetries of galaxy morphology:
rotation and reflection. In practice, this notebook uses a compact equivariant network over the **D4 symmetry group** (90° rotations + flips), which is the lightweight discrete setting most aligned with the augmentation choices above.


## 3.2 Improvement

<div class="alert alert-warning">

### 3.2.1 Task 2

Improve the CNN model (you can also improve based on other models, but CNN is currently the most common model for image recognition)

#### Option 1: Pre-trained ResNet18

ResNet18 is a convolutional neural network architecture from the ResNet family. '18' refers to 18-layer of this architecture. 

The model has been pretrained on ImageNet (1,000 classes) and learned useful generic features for images, such as edges, textures, shapes, etc. Fine-tuning allows us to leverage these features and adapt them to our own dataset.


#### Option 2: Attention layer

Relevant papers：

[CoAtNet: Marrying Convolution and Attention for All Data Sizes](https://arxiv.org/abs/2106.04803) 

[An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929). 

The later paper intorduced ViT model which is a transformer architecture used in vision tasks. 

Mathematically, convolution of $x$ with kernel matrix $w$ is:

$$y_i=\sum_{j \in \mathcal{L}(i)} w_{i-j} \odot x_j$$

where $\mathcal{L}(i)$ denotes a local neighbors

In vision tasks:

$$\text{standard self-attention matrix}=A_{ij}=\frac{\exp \left(x_i^{\top} x_j\right)}{\sum_{k \in \mathcal{G}} \exp \left(x_i^{\top} x_k\right)}$$


where $x_i$ represents the feature vector at spatial location $i$ in the image. 

Loosely speaking, the entries in attention matrix $A_{ij}$ means how important is position $j$ to position $i$.

Predict label $y$ using attention matrix:


$$y_i=\sum_{j \in \mathcal{G}} \frac{\exp \left(x_i^{\top} x_j\right)}{\sum_{k \in \mathcal{G}} \exp \left(x_i^{\top} x_k\right)} x_j$$

where $\mathcal{G}$ indicates the global spatial space. 


Combine Attention and convolution (pre-normalization):

$$\text{relative self-attention}=A_{i, j}=\sum_{k \in \mathcal{G}} \exp \left(x_i^{\top} x_k+\boldsymbol{w}_{i-k}\right)$$

So

$$y_i^{\mathrm{pre}}=\sum_{j \in \mathcal{G}} \frac{\exp \left(x_i^{\top} x_j+w_{i-j}\right)}{\sum_{k \in \mathcal{G}} \exp \left(x_i^{\top} x_k+w_{i-k}\right)} x_j$$


Further Improvment:

We can build multiple CNN layers to extract more 'meaningful' features, and use residual to prevent model deterioration. Reference paper: [ASTROFORMER: MORE DATA MIGHT NOT BE ALL YOU NEED FOR CLASSIFICATION](https://arxiv.org/abs/2304.05350)

(ASTROFORMER is the SOTA model for zoo2 galaxy classification)

#### Option 3: Others

Your own ideas are encouraged to achieve better performance.


</div>

In [ ]:

# ── 3.2.1 Improved model definition ─────────────────────────
# The improved model used in this notebook is a lightweight D4-equivariant E(2)-GCNN
# defined earlier from `model_source`.

model_improved = GalaxyModel(**model_kwargs).to(device)
model_improved.load_state_dict(torch.load('./MyImprovedModel.pth', map_location=device)['state_dict'])
print(model_improved)


In [ ]:

# ── 3.2.1 Improved model training ───────────────────────────
# Training was already completed in the main training section above.
# This cell keeps the notebook structure aligned with the original template.

print('Improved lightweight D4-equivariant E(2)-GCNN training has already been completed above.')
print('Saved file: ./MyImprovedModel.pth')


### 3.2.2 Evaluation on Local Validation Set

Please use an evaluation method similar to 3.1.2 to evaluate your model.  

As a baseline, using pre-trained ResNet18:

- **10% of whole dataset, 80/20 split** → ~65–70% accuracy  
- **30% of whole dataset, 80/20 split** → ~70% accuracy  
- **50% of whole dataset, 80/20 split** → ~75–80% accuracy  

Your solution should match or exceed these baselines for the dataset size you chose.

In [ ]:

# ── 3.2.2 Evaluate the improved model on the local validation set ────────────
model_improved.eval()

y_true_imp, y_pred_imp = [], []

with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        outputs = predict_with_tta(model_improved, imgs)
        preds = outputs.argmax(dim=1)

        y_true_imp.extend(lbls.cpu().numpy())
        y_pred_imp.extend(preds.cpu().numpy())

val_acc_imp = accuracy_score(y_true_imp, y_pred_imp)
val_macro_f1_imp = f1_score(y_true_imp, y_pred_imp, average='macro')

print(f'Improved Validation Accuracy : {val_acc_imp*100:.2f}%')
print(f'Improved Validation Macro-F1 : {val_macro_f1_imp:.4f}')
print(classification_report(y_true_imp, y_pred_imp, target_names=galaxy_classes, digits=4))


---
# 4. Final Evaluation on Held-out Test Set

Once you are happy with your model, run inference on the **fixed test set** (`X_test.npy`) provided by the instructor.

- You must **not** use `X_test.npy` in any way during training.
- Save your predictions to `y_pred_test.npy` and submit this file alongside your notebook.
- The instructor will compare your predictions against `y_test.npy` to compute your final score.

---

In [ ]:
%run run_inference.py \
    --test_h5 Galaxy10_DECals_20pct.h5 \
    --model_pth MyImprovedModel.pth \
    --transform_py my_preprocessing.py \
    --transform_name inference_transform

---
## 3.3 Classification by Human

<div class="alert alert-warning">

Look at the 25 galaxy images shown below (labels are hidden).  
Write your best guess for each galaxy class in the cell that follows.

</div>

In [ ]:
# NOTE: Read-only — do not modify

import random

random.seed(99)   # fixed seed so every student sees the same 25 images
sample_indices = [i + random.randint(0, 100) for i in range(25)]

fig = plt.figure(figsize=(20, 20))
for plot_i, img_idx in enumerate(sample_indices):
    plt.subplot(5, 5, plot_i + 1)
    plt.imshow(X_train[img_idx])
    plt.title(f'Image #{plot_i+1}', fontsize=9)
    # Labels intentionally hidden — plt.title(galaxy_classes[y_train[img_idx]])
    fig.tight_layout(pad=3.0)
plt.show()

<div class="alert alert-warning">

### ✏️ Your Human Classification Answers

Fill in your guesses below. Use the class names from `galaxy_classes`:

```
0: Disturbed Galaxies
1: Merging Galaxies
2: Round Smooth Galaxies
3: In-between Round Smooth Galaxies
4: Cigar Shaped Smooth Galaxies
5: Barred Spiral Galaxies
6: Unbarred Tight Spiral Galaxies
7: Unbarred Loose Spiral Galaxies
8: Edge-on Galaxies without Bulge
9: Edge-on Galaxies with Bulge
```

| Image # | Your Guess |
|---------|------------|
| 1  |  |
| 2  |  |
| 3  |  |
| 4  |  |
| 5  |  |
| 6  |  |
| 7  |  |
| 8  |  |
| 9  |  |
| 10 |  |
| 11 |  |
| 12 |  |
| 13 |  |
| 14 |  |
| 15 |  |
| 16 |  |
| 17 |  |
| 18 |  |
| 19 |  |
| 20 |  |
| 21 |  |
| 22 |  |
| 23 |  |
| 24 |  |
| 25 |  |

</div>